# jaxfne — Étude No. 8 · Continuous Drive Adaptation (HDP)

A ONE continuous simulation (not discrete trials, unlike Étude 9-12's
oddball/omission paradigms) demonstrating Homeostasis-Dependent Plasticity
(HDP) responding to a sustained step increase in drive: firing rate jumps,
then relaxes toward a new, lower, stable equilibrium as HDP's `H` factor
moves and mediates a real weight-scaling adaptation.

Uses the standalone `jaxfne.hdp_network` column path (`HDPColumnConfig` +
`simulate_edge_recurrent_izhikevich_hdp`), the SAME path validated this
session's `K_ctrl` stability fix against (20s/5-seed gate, 10/10 pass) —
not the `Configuration`/`.hdp()` wiring, which is separately queued
(plans.json `hdp-100k-100step-validation-run`) and not yet exercised.

## 1. Setup

**Deliberate `K_HDP` deviation from `DEFAULT_HDP`, explained up front:**
`DEFAULT_HDP`'s `K_HDP=0.01` is tuned conservative-for-stability (the fix
validated this session) — under it, `H` barely moves and the weight-
mediated adaptation this étude is about is not visible within a reasonable
demo duration. `K_HDP=0.1` (kept here, 20s duration) produces a real,
visible relaxation while `K_ctrl` stays at its validated `5.0` (the
stability-critical term is unchanged, only the weight-plasticity gain is
raised, and only modestly).

**Real methodology bug found and fixed while building this étude (not
papered over):** this repo has multiple registered Jupyter kernels, and
the *default* `python3` kernel has `PYTHONPATH` pointed at a **different,
unrelated jaxfne checkout** (`~/workspace/computational/jaxfne`), not this
repo. Executing this notebook's cells under that default kernel produced a
dramatic, reproducible runaway (`H` up to 7.96, firing rate pinned at 2000
Hz) for several `K_HDP` values that were rock-solid when run directly
against this repo's code — a genuine different-codebase discrepancy, not a
numerical sensitivity in this repo's HDP kernel. Re-running the identical
cells against the correct environment (the `jaxfne_analysis` kernel, or
equivalently this notebook's own `sys.path`) reproduced the safe, stable
result exactly. **If you re-run this notebook yourself, make sure your
kernel actually imports jaxfne from this checkout** — `import jaxfne;
print(jaxfne.__file__)` in the cell below confirms it.

In [1]:
import numpy as np
import jax
import jax.numpy as jnp
import jaxfne as jtfne
from jaxfne.hdp_network import HDPColumnConfig, build_model, DEFAULT_HDP, BASE_HDP_KWARGS_DEFAULT
from jaxfne.emitters import simulate_edge_recurrent_izhikevich_hdp

print("jaxfne", jtfne.__version__, jtfne.__file__)  # confirm this repo, not a stale checkout


jaxfne 0.4.4 /Users/hamednejat/workspace/analysis/jaxfne/jaxfne/__init__.py


## 2. Config

In [2]:
N_NEURONS = 100
SEED = 0
DT_MS = 0.5
DURATION_MS = 20000.0
STEP_FRACTION = 0.1     # perturbation onset, as a fraction of total duration
STEP_DRIVE = 3.0        # sustained extra drive from the step onward
K_HDP_DEMO = 0.1         # deliberately raised from DEFAULT_HDP's 0.01 -- see Étude intro

cfg = HDPColumnConfig(n_neurons=N_NEURONS, duration_ms=DURATION_MS, dt_ms=DT_MS, seed=SEED)
model = build_model(cfg)
n_steps = int(round(DURATION_MS / DT_MS))
n_neurons = model.params["emitter"].v0.shape[0]
step_at = int(n_steps * STEP_FRACTION)
print(f"n_steps={n_steps}, step onset at t={step_at * DT_MS:.0f}ms")


n_steps=40000, step onset at t=2000ms


## 3. Continuous drive schedule

`simulate_edge_recurrent_izhikevich_hdp`'s `drive_schedule` (shape
`(n_steps, n_neurons)`) is added to each neuron's baseline drive at every
timestep — a genuine continuous perturbation within ONE scan, not a
discrete-trial `StimulusSchedule` (Étude 9-12's mechanism).

In [3]:
drive_schedule = np.zeros((n_steps, n_neurons), dtype=np.float32)
drive_schedule[step_at:, :] = STEP_DRIVE
drive_schedule = jnp.asarray(drive_schedule)


## 4. Run

In [4]:
hdp_kwargs = dict(BASE_HDP_KWARGS_DEFAULT)
hdp_kwargs.update(DEFAULT_HDP)
hdp_kwargs["K_HDP"] = K_HDP_DEMO
print({k: hdp_kwargs[k] for k in ("K_HDP", "K_ctrl", "tau_0_ms")})

key = jax.random.PRNGKey(SEED)
voltages, spikes, sources, diag = simulate_edge_recurrent_izhikevich_hdp(
    model.params["emitter"], model.params["edge_list"], n_steps, DT_MS, key,
    drive_schedule=drive_schedule, **hdp_kwargs,
)
assert bool(jnp.all(jnp.isfinite(voltages))), "non-finite voltages -- do not trust this run"

H_trace = np.asarray(diag["H_trace"])
w_trace = np.asarray(diag["w_trace"])
spk = np.asarray(spikes)

WIN_MS = 500.0
win = int(WIN_MS / DT_MS)
window_starts = list(range(0, n_steps, win))
rate_by_window = np.array([spk[i:i + win].mean() * 1000.0 / DT_MS for i in window_starts])
H_mean_by_window = np.array([H_trace[i:i + win].mean() for i in window_starts])
for t0, rate, hmean in zip(window_starts, rate_by_window, H_mean_by_window):
    print(f"t={t0 * DT_MS:6.0f}ms  rate={rate:6.2f} Hz  H_mean={hmean:.4f}")


{'K_HDP': 0.1, 'K_ctrl': 5.0, 'tau_0_ms': 200.0}


t=     0ms  rate= 12.36 Hz  H_mean=1.0005
t=   500ms  rate= 11.14 Hz  H_mean=1.0008
t=  1000ms  rate= 11.32 Hz  H_mean=1.0010
t=  1500ms  rate= 10.70 Hz  H_mean=1.0011
t=  2000ms  rate= 25.54 Hz  H_mean=1.0012
t=  2500ms  rate= 24.00 Hz  H_mean=1.0012
t=  3000ms  rate= 25.50 Hz  H_mean=1.0013
t=  3500ms  rate= 24.00 Hz  H_mean=1.0014
t=  4000ms  rate= 24.00 Hz  H_mean=1.0015
t=  4500ms  rate= 25.54 Hz  H_mean=1.0015
t=  5000ms  rate= 24.02 Hz  H_mean=1.0015
t=  5500ms  rate= 24.06 Hz  H_mean=1.0016
t=  6000ms  rate= 24.14 Hz  H_mean=1.0016
t=  6500ms  rate= 24.78 Hz  H_mean=1.0016
t=  7000ms  rate= 24.98 Hz  H_mean=1.0016
t=  7500ms  rate= 24.04 Hz  H_mean=1.0016
t=  8000ms  rate= 24.16 Hz  H_mean=1.0016
t=  8500ms  rate= 24.20 Hz  H_mean=1.0016
t=  9000ms  rate= 24.22 Hz  H_mean=1.0016
t=  9500ms  rate= 24.26 Hz  H_mean=1.0016
t= 10000ms  rate= 24.20 Hz  H_mean=1.0016
t= 10500ms  rate= 24.20 Hz  H_mean=1.0016
t= 11000ms  rate= 23.96 Hz  H_mean=1.0016
t= 11500ms  rate= 23.58 Hz  H_mean

## 5. Objective — continuous adaptation magnitude

Three windows: `pre` (before the step), `peak` (the window right after the
step, before adaptation has had time to act), and `post` (the last window,
the new steady state after HDP has acted). A genuine continuous-adaptation
signature is `peak` well above `pre`, and `post` relaxed back down toward
(not necessarily all the way to) `pre` -- distinct from no-adaptation
(`post` staying at `peak`) or runaway instability (`post` diverging further).

In [5]:
pre_window_idx = (step_at // win) - 1
peak_window_idx = step_at // win
post_window_idx = -1

rate_pre = float(rate_by_window[pre_window_idx])
rate_peak = float(rate_by_window[peak_window_idx])
rate_post = float(rate_by_window[post_window_idx])
H_excursion = float(H_mean_by_window.min() - 1.0)  # most negative deviation from equilibrium

relaxation_fraction = (rate_peak - rate_post) / (rate_peak - rate_pre) if rate_peak != rate_pre else 0.0

print(f"rate_pre={rate_pre:.2f} Hz, rate_peak={rate_peak:.2f} Hz, rate_post={rate_post:.2f} Hz")
print(f"H excursion (min - 1.0): {H_excursion:+.4f}")
print(f"relaxation fraction (peak->post, of the peak-pre jump): {relaxation_fraction:.2f}")

# Computational diagnostic, not a claim of biological homeostatic validation --
# a same-model firing-rate/H trajectory under a deliberately-raised K_HDP,
# not a validated physiological adaptation timescale or magnitude.
assert np.isfinite(rate_pre) and np.isfinite(rate_peak) and np.isfinite(rate_post)
assert rate_peak > rate_pre, "expected a real rate jump at the drive step"


rate_pre=10.70 Hz, rate_peak=25.54 Hz, rate_post=13.28 Hz
H excursion (min - 1.0): -0.0020
relaxation fraction (peak->post, of the peak-pre jump): 0.83


## 6. Export

In [6]:
import json as _json
from pathlib import Path

OUT_DIR = Path("local/etude8")
OUT_DIR.mkdir(parents=True, exist_ok=True)

manifest = {
    "notebook": "jaxfne_etude_no_8_continuous_adaptation",
    "jaxfne_version": jtfne.__version__,
    "config": {
        "N_NEURONS": N_NEURONS, "DT_MS": DT_MS, "DURATION_MS": DURATION_MS, "SEED": SEED,
        "step_onset_ms": step_at * DT_MS, "step_drive": STEP_DRIVE,
        "K_HDP_demo": K_HDP_DEMO, "K_ctrl": hdp_kwargs["K_ctrl"], "tau_0_ms": hdp_kwargs["tau_0_ms"],
    },
    "results": {
        "rate_pre_hz": rate_pre, "rate_peak_hz": rate_peak, "rate_post_hz": rate_post,
        "H_excursion": H_excursion, "relaxation_fraction": relaxation_fraction,
    },
}
(OUT_DIR / "manifest.json").write_text(_json.dumps(manifest, indent=2))
print("wrote", OUT_DIR / "manifest.json")


wrote local/etude8/manifest.json
